In [1]:
import os, sys
dir2 = os.path.abspath('')
dir1 = os.path.dirname(dir2)
if not dir1 in sys.path: 
    sys.path.append(dir1)

from game.wordle import Wordle
from game.util import read_to_lines
from game.baseSolver import Solver
from game.rlSolver import RLSolver
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import random
from game.wordle import Wordle

In [23]:
guess_words = read_to_lines("../data/de_leipzig_5_letter.txt")
answer_words = read_to_lines("../data/de_wiktionary_5_letter_shortlist.txt")

de_config = {
    'max_guesses': 6,
    # The set of words that can potentially be solutions
    'candidate_set': answer_words,
    # The set of words that can be guessed validly
    'guess_set': guess_words
}


In [6]:
w = Wordle("notar", config=de_config)

In [7]:
w.guess("lösen")

LÖSEN
⬛⬛⬛⬛🟨


([0, 0, 0, 0, 1], 0)

In [24]:
s = RLSolver(config=de_config)

def random_solver_baseline(config, num_games=1000):
    """Pure random guessing - no learning"""
    wins = 0
    n_guesses = 0
    guess_dist = {i: 0 for i in range(1, 7)}
    guess_dist['failed'] = 0
    for _ in range(num_games):
        target = random.choice(config['candidate_set'])
        w = Wordle(target, config=de_config, verbose=False)
        candidates = list(config['candidate_set'])
        #print(f"guessing word {target}")
        
        for guess_num in range(6):
            guess = random.choice(candidates)
            if guess == target:
                wins += 1
                n_guesses += guess_num + 1
                guess_dist[guess_num + 1] += 1
                break
            
            # Filter candidates based on clue
            clue, _ = w.guess(guess)
            candidates = [c for c in candidates if s._is_valid_candidate(c, guess, clue)]
            
            if guess_num == 5:
                guess_dist["failed"] += 1 

    
    return wins / num_games, n_guesses / wins, guess_dist

win_rate, avg_guess, guess_dist = random_solver_baseline(de_config)
print(f"win rate is {win_rate}")
print(f"average number of guesses is {avg_guess}")

win rate is 0.951
average number of guesses is 4.279705573080967


In [25]:
guess_dist

{1: 1, 2: 26, 3: 187, 4: 339, 5: 288, 6: 110, 'failed': 49}